### 불용어 파일 생성

In [ ]:
core_entropy

In [4]:
import pandas as pd
from kneed import KneeLocator
import matplotlib.pyplot as plt
from matplotlib import rc
import os

# ---------------------------------------------------------
# 1. 설정 및 폴더 생성
# ---------------------------------------------------------
INPUT_FILE = "legal_stopword_analysis_final.csv"
OUTPUT_DIR = "stopword_공통"  # 판례 및 통합 데이터 결과 저장소

os.makedirs(OUTPUT_DIR, exist_ok=True)
rc('font', family='AppleGothic')
plt.rcParams['axes.unicode_minus'] = False

if not os.path.exists(INPUT_FILE):
    print(f"[Error] '{INPUT_FILE}' 파일이 없습니다.")
    exit()
    
df = pd.read_csv(INPUT_FILE)

# ---------------------------------------------------------
# 함수: 엘보우 포인트 계산 (수정됨)
# ---------------------------------------------------------
def get_elbow_candidates(df, target_col, method_name, filename_suffix, save_dir):
    # 정렬
    sorted_df = df.sort_values(by=target_col, ascending=False).reset_index(drop=True)
    
    x = sorted_df.index.tolist()
    y = sorted_df[target_col].tolist()
    
    # Kneedle 알고리즘
    kneedle = KneeLocator(x, y, curve="convex", direction="decreasing", S=1.0)
    elbow_rank = kneedle.elbow
    
    # 엘보우 못 찾았을 때 대비
    if elbow_rank is None:
        print(f"[{method_name}] 엘보우 포인트를 찾지 못했습니다. Top 100개로 대체합니다.")
        elbow_rank = 100
    
    # [수정된 부분] 변수명 통일 (elbow_value)
    elbow_value = y[elbow_rank] 
    elbow_word = sorted_df.iloc[elbow_rank]['word']
    
    candidates = sorted_df.head(elbow_rank).copy()
    candidates['selected_by'] = method_name 
    
    print(f"[{method_name}] Cut-off: Top {elbow_rank}위 ('{elbow_word}', 점수: {elbow_value:.4f})")
    
    # 그래프 그리기
    plt.figure(figsize=(10, 6))
    plt.plot(x, y, label=f'{method_name} Score')
    plt.vlines(elbow_rank, plt.ylim()[0], plt.ylim()[1], linestyles='dashed', colors='red', label='Elbow Point')
    
    # [오류 해결] 이제 elbow_value가 정의되었으므로 에러 안 남
    plt.scatter([elbow_rank], [elbow_value], color='red', s=100, zorder=5)
    
    plt.title(f"[공통] {method_name} Elbow Point")
    plt.xlabel("Rank")
    plt.ylabel("Score")
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    plt.annotate(f'Cut-off: {elbow_rank}위\nWord: {elbow_word}', 
                 xy=(elbow_rank, elbow_value), 
                 xytext=(elbow_rank + 50, elbow_value + 0.05),
                 arrowprops=dict(facecolor='black', shrink=0.05))
    
    # 이미지 저장
    img_path = os.path.join(save_dir, f"graph_{filename_suffix}.png")
    plt.savefig(img_path)
    plt.close() 
    
    return candidates

# ---------------------------------------------------------
# 3. 세 가지 관점에서 후보 추출 (다시 실행)
# ---------------------------------------------------------
print(">>> 1. 단순 빈도(Zipf) 기반 추출 중...")
cand_tf = get_elbow_candidates(df, 'count', 'Frequency_Zipf', 'A_zipf', OUTPUT_DIR)

print(">>> 2. 엔트로피(Entropy) 기반 추출 중...")
cand_ent = get_elbow_candidates(df, 'score_entropy', 'Entropy_Randomness', 'B_entropy', OUTPUT_DIR)

print(">>> 3. 안정성(MP-VP) 기반 추출 중...")
cand_mpvp = get_elbow_candidates(df, 'score_mpvp', 'Stability_MPVP', 'C_mpvp', OUTPUT_DIR)

# ---------------------------------------------------------
# 4. 결과 통합 (Union & Intersection)
# ---------------------------------------------------------
# (1) Union (합집합 - 실험 D)
all_candidates = pd.concat([cand_tf, cand_ent, cand_mpvp])
final_df = all_candidates.groupby('word').agg({
    'selected_by': lambda x: ', '.join(set(x)), 
    'count': 'first',
    'score_entropy': 'first',
    'score_mpvp': 'first'
}).reset_index().sort_values(by='count', ascending=False)

# (2) Intersection (교집합 - 실험 E, ★Golden List★)
set_tf = set(cand_tf['word'])
set_ent = set(cand_ent['word'])
set_mpvp = set(cand_mpvp['word'])
set_intersection = set_tf.intersection(set_ent).intersection(set_mpvp)
cand_intersection = final_df[final_df['word'].isin(set_intersection)].copy()

# ---------------------------------------------------------
# 5. 저장 1: 변호사 공유용 (Excel)
# ---------------------------------------------------------
excel_path = os.path.join(OUTPUT_DIR, "불용어_후보_검토요청서.xlsx")

excel_df = final_df[['word', 'selected_by']].copy()
excel_df.columns = ['단어(형태소)', '추출_알고리즘_근거']
excel_df['추출_알고리즘_근거'] = excel_df['추출_알고리즘_근거'].replace({
    'Frequency_Zipf': '고빈도 단어',
    'Entropy_Randomness': '구조적 불용어',
    'Stability_MPVP': '기능적 불용어'
}, regex=True)
excel_df['삭제여부(O/X)'] = '' 
excel_df['비고(의견)'] = ''

with pd.ExcelWriter(excel_path) as writer:
    excel_df.to_excel(writer, sheet_name='종합_후보_목록', index=False)
    cand_intersection.to_excel(writer, sheet_name='추천_교집합_핵심', index=False)
    cand_tf.to_excel(writer, sheet_name='참고_빈도기반', index=False)
    cand_ent.to_excel(writer, sheet_name='참고_엔트로피', index=False)
    cand_mpvp.to_excel(writer, sheet_name='참고_안정성', index=False)

print(f"변호사 공유용 엑셀 저장 완료: {excel_path}")

# ---------------------------------------------------------
# 6. 저장 2: 연구자용 (CSV)
# ---------------------------------------------------------
print("\n>>> 연구 실험용 CSV 파일 분리 저장 중...")

cand_tf.to_csv(os.path.join(OUTPUT_DIR, "stopwords_exp_A_zipf.csv"), index=False, encoding='utf-8-sig')
cand_ent.to_csv(os.path.join(OUTPUT_DIR, "stopwords_exp_B_entropy.csv"), index=False, encoding='utf-8-sig')
cand_mpvp.to_csv(os.path.join(OUTPUT_DIR, "stopwords_exp_C_mpvp.csv"), index=False, encoding='utf-8-sig')
final_df.to_csv(os.path.join(OUTPUT_DIR, "stopwords_exp_D_union.csv"), index=False, encoding='utf-8-sig')
cand_intersection.to_csv(os.path.join(OUTPUT_DIR, "stopwords_exp_E_intersection.csv"), index=False, encoding='utf-8-sig')

print(f"\n모든 파일 저장 완료! (폴더: {OUTPUT_DIR})")
print(f" - 교집합(Exp E) 개수: {len(cand_intersection)}개")

>>> 1. 단순 빈도(Zipf) 기반 추출 중...
[Frequency_Zipf] Cut-off: Top 167위 ('책임', 점수: 9717.0000)
>>> 2. 엔트로피(Entropy) 기반 추출 중...
[Entropy_Randomness] Cut-off: Top 226위 ('기본', 점수: 0.7885)
>>> 3. 안정성(MP-VP) 기반 추출 중...
[Stability_MPVP] Cut-off: Top 54위 ('담당', 점수: 1.0646)
변호사 공유용 엑셀 저장 완료: stopword_공통/불용어_후보_검토요청서.xlsx

>>> 연구 실험용 CSV 파일 분리 저장 중...

모든 파일 저장 완료! (폴더: stopword_공통)
 - 교집합(Exp E) 개수: 44개


### MP, VP, IF, IDF 다 추출(폐기)

In [5]:
import pandas as pd
from kneed import KneeLocator
import matplotlib.pyplot as plt
from matplotlib import rc
import os

# ---------------------------------------------------------
# 1. 설정 및 데이터 로드
# ---------------------------------------------------------
INPUT_FILE = "legal_stopword_analysis_final.csv"
OUTPUT_DIR = "stopword_공통_상세분석"  # 폴더명 변경 (상세분석)

os.makedirs(OUTPUT_DIR, exist_ok=True)
rc('font', family='AppleGothic')
plt.rcParams['axes.unicode_minus'] = False

if not os.path.exists(INPUT_FILE):
    print(f"[Error] '{INPUT_FILE}' 파일이 없습니다.")
    exit()
    
df = pd.read_csv(INPUT_FILE)

# ---------------------------------------------------------
# 2. 단독 지표 점수화 (Preprocessing)
# ---------------------------------------------------------
# 정규화 함수 (Min-Max)
def normalize(series):
    return (series - series.min()) / (series.max() - series.min())

# (1) TF (Count): 높을수록 불용어 -> 그대로 사용
# (2) IDF (TF-IDF): 낮을수록 불용어 -> Score = 1 - Norm(TF-IDF) (기존 score_tfidf 사용)
# (3) Entropy: 높을수록 불용어 -> 그대로 사용 (기존 score_entropy 사용)

# (4) MP (Mean Probability) 단독: 높을수록 불용어
df['score_mp_only'] = normalize(df['mp'])

# (5) VP (Variance) 단독: 낮을수록(안정적일수록) 불용어 -> 뒤집어야 함
df['score_vp_inv'] = 1 - normalize(df['vp'])

# (6) MP-VP 복합: 기존 score_mpvp 사용

print(">>> 단독 지표 스코어 계산 완료.")

# ---------------------------------------------------------
# 3. 엘보우 포인트 계산 함수
# ---------------------------------------------------------
def get_elbow_candidates(df, target_col, method_name, filename_suffix, save_dir):
    # 정렬 (점수가 높은 순서대로)
    sorted_df = df.sort_values(by=target_col, ascending=False).reset_index(drop=True)
    
    x = sorted_df.index.tolist()
    y = sorted_df[target_col].tolist()
    
    # Kneedle
    kneedle = KneeLocator(x, y, curve="convex", direction="decreasing", S=1.0)
    elbow_rank = kneedle.elbow
    
    if elbow_rank is None:
        elbow_rank = 50
    
    elbow_value = y[elbow_rank]
    elbow_word = sorted_df.iloc[elbow_rank]['word']
    
    candidates = sorted_df.head(elbow_rank).copy()
    candidates['selected_by'] = method_name 
    
    print(f"[{method_name}] Cut-off: Top {elbow_rank}위 ('{elbow_word}', 점수: {elbow_value:.4f})")
    
    # 그래프
    plt.figure(figsize=(10, 6))
    plt.plot(x, y, label=f'{method_name} Score')
    plt.vlines(elbow_rank, plt.ylim()[0], plt.ylim()[1], linestyles='dashed', colors='red', label='Elbow Point')
    plt.scatter([elbow_rank], [elbow_value], color='red', s=100, zorder=5)
    plt.title(f"{method_name} Elbow Point")
    plt.xlabel("Rank")
    plt.ylabel("Score")
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    img_path = os.path.join(save_dir, f"graph_{filename_suffix}.png")
    plt.savefig(img_path)
    plt.close()
    
    return candidates

# ---------------------------------------------------------
# 4. 6가지 관점 후보 추출 (Full Scanning)
# ---------------------------------------------------------
print("\n>>> [1/6] TF(빈도) 단독 분석...")
cand_tf = get_elbow_candidates(df, 'count', 'Only_TF', '1_TF', OUTPUT_DIR)

print(">>> [2/6] IDF(가중치) 단독 분석 (Low IDF)...")
cand_idf = get_elbow_candidates(df, 'score_tfidf', 'Only_IDF', '2_IDF', OUTPUT_DIR)

print(">>> [3/6] MP(평균확률) 단독 분석...")
cand_mp = get_elbow_candidates(df, 'score_mp_only', 'Only_MP', '3_MP', OUTPUT_DIR)

print(">>> [4/6] VP(분산) 단독 분석 (Low Variance)...")
cand_vp = get_elbow_candidates(df, 'score_vp_inv', 'Only_VP', '4_VP', OUTPUT_DIR)

print(">>> [5/6] Entropy(엔트로피) 분석...")
cand_ent = get_elbow_candidates(df, 'score_entropy', 'Entropy', '5_Entropy', OUTPUT_DIR)

print(">>> [6/6] MP-VP(복합안정성) 분석...")
cand_mpvp = get_elbow_candidates(df, 'score_mpvp', 'Composite_MPVP', '6_MPVP', OUTPUT_DIR)

# ---------------------------------------------------------
# 5. 결과 통합 및 저장
# ---------------------------------------------------------
# 모든 후보 합치기 (Union)
all_candidates = pd.concat([cand_tf, cand_idf, cand_mp, cand_vp, cand_ent, cand_mpvp])

# 종합 테이블 생성
final_df = all_candidates.groupby('word').agg({
    'selected_by': lambda x: ', '.join(sorted(list(set(x)))), # 어떤 기법들에 걸렸는지
    'count': 'first'
}).reset_index().sort_values(by='count', ascending=False)

# 교집합 (Golden List) - 이번엔 '6개 중 3개 이상' 같은 보팅(Voting)도 가능하지만, 
# 일단 기존의 핵심(TF, Ent, MPVP) 교집합을 메인으로 유지하거나, 
# 전체 교집합을 볼 수도 있음. 여기선 '전체 Union'을 저장하고 개별 파일 제공.

# ---------------------------------------------------------
# 6. 엑셀 저장 (탭으로 구분)
# ---------------------------------------------------------
excel_path = os.path.join(OUTPUT_DIR, "불용어_상세분석_리포트.xlsx")

with pd.ExcelWriter(excel_path) as writer:
    # 종합
    final_df.to_excel(writer, sheet_name='종합_후보_목록(Union)', index=False)
    
    # 단독 지표
    cand_tf.to_excel(writer, sheet_name='1_TF_단독', index=False)
    cand_idf.to_excel(writer, sheet_name='2_IDF_단독', index=False)
    cand_mp.to_excel(writer, sheet_name='3_MP_단독', index=False)
    cand_vp.to_excel(writer, sheet_name='4_VP_단독', index=False)
    
    # 기존/복합 지표
    cand_ent.to_excel(writer, sheet_name='5_Entropy', index=False)
    cand_mpvp.to_excel(writer, sheet_name='6_MPVP_복합', index=False)

print(f"\n엑셀 리포트 저장 완료: {excel_path}")

# ---------------------------------------------------------
# 7. CSV 개별 저장 (실험용)
# ---------------------------------------------------------
cand_tf.to_csv(os.path.join(OUTPUT_DIR, "stopwords_TF.csv"), index=False)
cand_idf.to_csv(os.path.join(OUTPUT_DIR, "stopwords_IDF.csv"), index=False)
cand_mp.to_csv(os.path.join(OUTPUT_DIR, "stopwords_MP.csv"), index=False)
cand_vp.to_csv(os.path.join(OUTPUT_DIR, "stopwords_VP.csv"), index=False)
cand_ent.to_csv(os.path.join(OUTPUT_DIR, "stopwords_Entropy.csv"), index=False)
cand_mpvp.to_csv(os.path.join(OUTPUT_DIR, "stopwords_MPVP.csv"), index=False)

print(f"개별 CSV 파일 저장 완료 (폴더: {OUTPUT_DIR})")
print("이제 LBOX 벤치마크 코드에서 이 파일들을 추가해서 돌려보시면 됩니다!")

>>> 단독 지표 스코어 계산 완료.

>>> [1/6] TF(빈도) 단독 분석...
[Only_TF] Cut-off: Top 167위 ('책임', 점수: 9717.0000)
>>> [2/6] IDF(가중치) 단독 분석 (Low IDF)...
[Only_IDF] Cut-off: Top 0위 ('續行', 점수: 1.0000)
>>> [3/6] MP(평균확률) 단독 분석...
[Only_MP] Cut-off: Top 34위 ('청구', 점수: 0.2011)
>>> [4/6] VP(분산) 단독 분석 (Low Variance)...
[Only_VP] Cut-off: Top 0위 ('봉안', 점수: 1.0000)
>>> [5/6] Entropy(엔트로피) 분석...
[Entropy] Cut-off: Top 226위 ('기본', 점수: 0.7885)
>>> [6/6] MP-VP(복합안정성) 분석...
[Composite_MPVP] Cut-off: Top 54위 ('담당', 점수: 1.0646)

엑셀 리포트 저장 완료: stopword_공통_상세분석/불용어_상세분석_리포트.xlsx
개별 CSV 파일 저장 완료 (폴더: stopword_공통_상세분석)
이제 LBOX 벤치마크 코드에서 이 파일들을 추가해서 돌려보시면 됩니다!
